# TP2: Prédiction de la Structure Secondaire des Protéines

## Introduction

Ce notebook implémente un système de prédiction de la structure secondaire des protéines (hélice α, feuillet β, ou coil) en utilisant une approche par **fenêtres glissantes** combinée à un **MLP (Perceptron Multicouches)**.

### Problème biologique
- Une protéine est une séquence d'acides aminés (20 types différents)
- La structure secondaire correspond à la conformation locale de la chaîne (hélice, feuillet, ou désordonné)
- C'est un problème de classification multi-classes (3 classes)

### Approche
- **Fenêtre glissante** : Pour prédire la structure d'un résidu, on regarde ses voisins (fenêtre de 13 acides aminés)
- **One-Hot Encoding** : Chaque acide aminé est encodé comme un vecteur binaire de dimension 21
- **MLP** : Réseau de neurones

In [1]:
# =============================================================================
#                       IMPORTATION DES BIBLIOTHÈQUES
# =============================================================================
#
# - torch          : Le framework de deep learning (PyTorch)
# - torch.nn       : Les modules de réseaux de neurones
# - torch.nn.functional : Les fonctions utilitaires (comme softmax, one-hot)
# - numpy : Pour les calculs numériques
# - pandas : Pour manipuler les données tabulaires
# - matplotlib & seaborn : Pour visualiser les résultats
# - sklearn : Pour les métriques et splits de données

import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split

---

## 1. Chargement et Prétraitement des Données

### 1.1 Format des données

Les données sont au format suivant :
```
<>           <- Marque le début d'une nouvelle protéine
A h          <- Acide aminé 'A' (Alanine) a une structure 'h' (hélice)
C e          <- Acide aminé 'C' (Cystéine) a une structure 'e' (feuillet)
E _          <- Acide aminé 'E' a une structure '_' (coil)
...
```

### 1.2 Dictionnaires de mapping

On crée des dictionnaires pour convertir :
- Les acide aminés en indices (0-19) + PAD (20)
- Les labels de structure en indices (0, 1, 2)

In [2]:
### ==========================================================================
#                    FONCTIONS DE CHARGEMENT DES DONNÉES
# ==========================================================================
# Cette fonction lit le contenu d'un fichier texte brut
# Elle retourne simplement une chaîne de caractères avec tout le contenu
# 
# Args:
#     data_name (str) : Le chemin vers le fichier de données
# 
# Returns:
#     str : Le contenu brut du fichier

def get_data(data_name):
    """
    Charge les données brutes depuis un fichier texte.
    
    Cette fonction ouvre le fichier en mode lecture et retourne
    l'ensemble du contenu sous forme d'une seule chaîne de caractères.
    """
    with open(data_name,'r') as data:
        return data.read()

In [3]:
# =============================================================================
#                       DICTIONNAIRES DE MAPPING
# =============================================================================
# 
# Ces dictionnaires permettent de convertir les données textuelles en nombres
# afin que notre réseau de neurones puisse les traiter.
#
# AA_MAP : Conversion Acide Aminé -> Indice numérique
#   - 20 acide aminés standard (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y)
#   - '<PAD>' (index 20) : Marqueur de remplissage pour les bords des séquences
#
# LABEL_MAP : Conversion Structure Secondaire -> Indice
#   - 'h' (0) : Hélice α (alpha helix)
#   - 'e' (1) : Feuillet β (beta sheet)  
#   - '_' (2) : Coil

# 1. Mapping des Acides Aminés (0-20, où 20 est le PAD)
# Voici les 20 acide aminés standards + le marqueur de padding
AA_MAP = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '<PAD>': 20  # Le Spacer pour les bords (marqueur spécial)
}

# 2. Mapping des labels de structure secondaire
# h = Hélice alpha, e = Feuillet bêta, _ = Coil
LABEL_MAP = {'h': 0, 'e': 1, '_': 2}

### 1.3 Parsing des données brutes

La fonction `parse_raw_data` transforme le texte brut en une liste de protéines structurées.

**Logique de parsing :**
1. On parcourt le fichier ligne par ligne
2. `<>` indique le début d'une nouvelle protéine
3. Chaque ligne contient un acide aminé et son label (ex: "E _")
4. On convertit immédiatement en indices numériques

In [4]:
# =============================================================================
#                    FONCTION DE PARSING DES DONNÉES
# =============================================================================
#
# Cette fonction prend le texte brut et le transforme en une liste de protéines.
#
# Format d'entrée (raw_text):
# <>               <- Début de protéine
# A h              <- Acide aminé A, structure h (hélice)
# C _              <- Acide aminé C, structure _ (coil)
# ...
# <>               <- Début de la prochaine protéine
# ...
#
# Format de sortie (proteins):
# [
#   ([0, 1, 2, ...], [0, 2, 1, ...]),  # (séquence_aa, séquence_labels)
#   ([...], [...]),
#   ...
# ]

def parse_raw_data(raw_text):
    """
    Lit le texte brut et retourne une liste de protéines.
    Chaque protéine est un tuple (liste_indices_aa, liste_indices_labels)
    
    Args:
        raw_text (str): Le texte brut contenant toutes les protéines
    
    Returns:
        list: Liste de tuples (seq_aa, seq_labels) pour chaque protéine
    """
    # Initialisation des listes pour stocker les données
    proteins = []
    current_aa_seq = []      # Liste des indices d'acides aminés pour la protéine courante
    current_label_seq = []   # Liste des indices de labels pour la protéine courante
    
    # On sépare le texte en lignes et on parcourt chaque ligne
    lines = raw_text.strip().split('\n')
    
    for line in lines:
        # On enlève les espaces au début et à la fin
        line = line.strip()
        if not line: continue  # On ignore les lignes vides
        
        # Le marqueur '<>' indique le début d'une nouvelle protéine
        if line == '<>':
            # Nouvelle protéine : on sauvegarde la précédente si elle existe
            # (on vérifie avec 'if current_aa_seq' pour éviter une proteine vide au début)
            if current_aa_seq:
                proteins.append((current_aa_seq, current_label_seq))
            
            # On réinitialise les listes pour la nouvelle protéine
            current_aa_seq = []
            current_label_seq = []
        else:
            # Ligne de donnée : "E _" ou "F h" (acide aminé + label)
            parts = line.split()
            if len(parts) == 2:  # Vérification qu'on a bien 2 éléments
                aa, label = parts[0], parts[1]
                
                # Conversion immédiate en entiers grâce à nos dictionnaires
                # Si l'acide aminé est dans notre dictionnaire, on le convertit
                if aa in AA_MAP:
                    current_aa_seq.append(AA_MAP[aa])
                    current_label_seq.append(LABEL_MAP[label])
    
    # Ne pas oublier de sauvegarder la DERNIÈRE protéine du fichier
    # (car elle n'est pas suivie par '<>')
    if current_aa_seq:
        proteins.append((current_aa_seq, current_label_seq))
    
    return proteins

### 1.4 Création des fenêtres glissantes

**Pourquoi des fenêtres ?**

Pour prédire la structure d'un acide aminé spécifique, il faut tenir compte de son **contexte** (ses voisins). On utilise une fenêtre centrée sur le résidu à prédire.

**Exemple avec fenêtre de taille 13 :**
```
Séquence: A C D E F G H I ...
              ^
              |__ Résidu central (index 6 dans la fenêtre)
Fenêtre: [PAD, PAD, PAD, PAD, PAD, PAD, A, C, D, E, F, G, H]
```

**Padding :** On ajoute des marqueurs spéciaux (PAD) au début et à la fin pour pouvoir traiter les premiers et derniers résidus.

In [5]:
# =============================================================================
#               FONCTION DE CRÉATION DE FENÊTRES GLISSANTES
# =============================================================================
#
# Cette fonction transforme les séquences complètes en un ensemble de fenêtres.
#
# Idée principale :
# - Pour CHAQUE résidu de la protéine, on crée une fenêtre de taille fixe
# - Cette fenêtre contient le résidu central + ses voisins
# - On "décale" la fenêtre d'un cran à chaque fois (glissante)
#
# Args:
#     proteins (list): Liste de tuples (seq_aa, seq_labels)
#     window_size (int): Taille de la fenêtre (défaut=13, donc 6 voisins de chaque côté)
#
# Returns:
#     X (torch.Tensor): Tenseur de forme [n_samples, window_size] avec les indices
#     Y (torch.Tensor): Tenseur de forme [n_samples] avec les labels

def create_windows(proteins, window_size=13):
    """
    Transforme les séquences complètes en fenêtres glissantes.
    
    Pour chaque résidu d'une protéine, on crée une fenêtre 
    ce résidu et ses voisins. Cela permet au modèle d'apprendre le contexte local.
    """
    
    # half_window = 6 pour window_size=13
    # Cela nous donne 6 éléments à gauche + 1 central + 6 éléments à droite
    half_window = window_size // 2
    
    X = []  # Stockera toutes les fenêtres d'entrée
    Y = []  # Stockera tous les labels cibles
    
    # On parcourt chaque protéine
    for aa_seq, label_seq in proteins:
        # L = longueur de cette protéine
        L = len(aa_seq)
        
        # =================================================================
        # PHASE DE PADDING
        # =================================================================
        # On ajoute des marqueurs PAD au début et à la fin de la séquence
        # Cela permet de traiter les premiers et derniers résidus de la protéine
        # sans avoir d'erreur d'index.
        #
        # Exemple avec half_window=6:
        # Original: [A, C, D, E, F]
        # Paddé:    [PAD, PAD, PAD, PAD, PAD, PAD, A, C, D, E, F, PAD, PAD, PAD, PAD, PAD, PAD]
        padded_seq = [AA_MAP['<PAD>']] * half_window + aa_seq + [AA_MAP['<PAD>']] * half_window
        
        # =================================================================
        # CRÉATION DES FENÊTRES (GLISSANTES)
        # =================================================================
        # On parcourt la séquence et on crée une fenêtre pour chaque position
        for i in range(L):
            # i correspond à la position du résidu central dans la séquence originale
            # Dans padded_seq, le résidu central est à l'index i + half_window
            # (à cause du padding qu'on a ajouté au début)
            
            # la fenêtre de taille window_size à partir de la position centrale
            window = padded_seq[i : i + window_size]
            
            # Le label cible est celui du résidu central (le i-ème résidu)
            target = label_seq[i]
            
            # On ajoute cette fenêtre et son label à nos listes
            X.append(window)
            Y.append(target)
    
    # Conversion en tenseurs PyTorch
    # X: tenseur d'indices (long) de forme [n_windows, window_size]
    # Y: tenseur de labels (long) de forme [n_windows]
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

### 1.5 Encodage One-Hot

Les réseaux de neurones ne peuvent pas traiter directement des indices entiers. 
On utilise l'**encodage One-Hot** pour convertir chaque indice en vecteur binaire.

**Exemple :**
- Indice 0 (Alanine) → [1, 0, 0, 0, ..., 0] (21 dimensions)
- Indice 1 (Cystéine) → [0, 1, 0, 0, ..., 0]

Une fenêtre de 13 acide aminés devient donc un vecteur de dimension 13 × 21 = 273.

In [6]:
# =============================================================================
#                    FONCTION D'ENCODAGE ONE-HOT
# =============================================================================
#
# Cette fonction convertit les indices d'acides aminés en vecteurs one-hot.
#
# Args:
#     data (torch.Tensor): Tenseur d'indices de forme [Batch, Window_Size]
#                         Exemple: [[0, 1, 2, ..., 12], [5, 3, 1, ..., 8], ...]
#
# Returns:
#     torch.Tensor: Tenseur one-hot de forme [Batch, Window_Size * 21]
#                  après aplatissage
#

# X_batch contient des entiers [Batch_Size, 13]
# On convertit en One-Hot float : [Batch_Size, 13, 21]
def one_hot(data):
    """
    Convertit les indices en encodage one-hot.
    
    Étapes:
    1. F.one_hot(data, num_classes=21) crée un tenseur de forme [N, 13, 21]
    2. .float() convertit les booléens en flottants
    3. .reshape() aplatit pour obtenir [N, 273] (13 * 21 = 273)
    """
    # Étape 1: One-hot encoding avec 21 classes (20 AA + PAD)
    # Résultat: [Batch, Window, 21] = [N, 13, 21]
    X_one_hot = F.one_hot(data, num_classes=21).float() 
    
    # On affiche la forme pour vérification
    print(f"Shape après one-hot: {X_one_hot.shape}")  # Devrait être [N, 13, 21]
    
    # Étape 2: Aplatissage (flatten)
    # Le MLP attend un vecteur d'entrée de taille固定
    # 13窗口 * 21特征 = 273输入特征
    X_one_hot = X_one_hot.reshape(X_one_hot.shape[0], X_one_hot.shape[1] * X_one_hot.shape[2])
    print(f"Shape après flatten: {X_one_hot.shape}")  # Devrait être [N, 273]
    
    return X_one_hot

---

## 2 Chargement des Données (Train/Val/Test)

Maintenant, on charge les trois ensembles de données :
- **Train** : Pour l'entraînement du modèle
- **Validation** : Pour sélectionner les hyperparamètres
- **Test** : Pour l'évaluation finale

### Fonction utilisée pour créer l'ensemble de validation (val)

In [7]:
"""
#Fonction pour écrire les protéines dans un fichier

def write_proteins_to_file(proteins, filename):
    with open(filename, 'w') as f:
        for aa_seq, label_seq in proteins:
            f.write('<>\n')
            for aa_idx, label_idx in zip(aa_seq, label_seq):
                aa = [k for k, v in AA_MAP.items() if v == aa_idx and k != '<PAD>'][0]
                label = [k for k, v in LABEL_MAP.items() if v == label_idx][0]
                f.write(f'{aa} {label}\n')
            f.write('<end>\n')

# Créer le fichier de validation
raw_data_sample = get_data('protein-secondary-structure-real.train')

data = parse_raw_data(raw_data_sample)
print(f"Nombre de protéines trouvées : {len(data)}")

# Split des protéines en train et validation
train_proteins, val_proteins = train_test_split(data, test_size=0.1, random_state=42)


write_proteins_to_file(val_proteins, 'protein-secondary-structure.val')
write_proteins_to_file(train_proteins, 'protein-secondary-structure.train')

print("Fichier 'protein-secondary-structure.val' créé avec les protéines de validation.")

"""

'\n#Fonction pour écrire les protéines dans un fichier\n\ndef write_proteins_to_file(proteins, filename):\n    with open(filename, \'w\') as f:\n        for aa_seq, label_seq in proteins:\n            f.write(\'<>\n\')\n            for aa_idx, label_idx in zip(aa_seq, label_seq):\n                aa = [k for k, v in AA_MAP.items() if v == aa_idx and k != \'<PAD>\'][0]\n                label = [k for k, v in LABEL_MAP.items() if v == label_idx][0]\n                f.write(f\'{aa} {label}\n\')\n            f.write(\'<end>\n\')\n\n# Créer le fichier de validation\nraw_data_sample = get_data(\'protein-secondary-structure-real.train\')\n\ndata = parse_raw_data(raw_data_sample)\nprint(f"Nombre de protéines trouvées : {len(data)}")\n\n# Split des protéines en train et validation\ntrain_proteins, val_proteins = train_test_split(data, test_size=0.1, random_state=42)\n\n\nwrite_proteins_to_file(val_proteins, \'protein-secondary-structure.val\')\nwrite_proteins_to_file(train_proteins, \'protein-s

In [8]:
# =============================================================================
#              CHARGEMENT DES DONNÉES D'ENTRAÎNEMENT
# =============================================================================
#
# Étapes:
# 1. Lire le fichier train avec get_data()
# 2. Parser avec parse_raw_data() pour obtenir les protéines
# 3. Créer les fenêtres avec create_windows()
#
# Résultat: X_true (fenêtres) et y_true (labels)

# 1. Chargement du fichier brut
raw_data_sample = get_data('protein-secondary-structure.train')

# 2. Parsing des données
data_train = parse_raw_data(raw_data_sample)
print(f"Nombre de protéines dans le train: {len(data_train)}")

# 3. Création des fenêtres glissantes
# Pour chaque résidu, on crée une fenêtre de 13 acide aminés
X_true, y_true = create_windows(data_train, window_size=13)

# Vérification des données
print(f"\n--- Exemple de données d'entraînement ---")
print(f"Fenêtre X[0]: {X_true[0]}")
print(f"Indices: {X_true[0].tolist()}")
print(f"Label y[0]: {y_true[0]} (0=h, 1=e, 2=_)")
print(f"Forme totale X train: {X_true.shape}")  # Devrait être [N_train, 13]

Nombre de protéines dans le train: 99

--- Exemple de données d'entraînement ---
Fenêtre X[0]: tensor([20, 20, 20, 20, 20, 20,  5, 17,  5, 16, 17, 12, 10])
Indices: [20, 20, 20, 20, 20, 20, 5, 17, 5, 16, 17, 12, 10]
Label y[0]: 2 (0=h, 1=e, 2=_)
Forme totale X train: torch.Size([16072, 13])


In [9]:
# =============================================================================
#              CHARGEMENT DES DONNÉES DE VALIDATION
# =============================================================================

# Même processus que pour le train
raw_data_sample = get_data('protein-secondary-structure.val')
val_train = parse_raw_data(raw_data_sample)
X_val, y_val = create_windows(val_train, window_size=13)

print(f"Nombre de protéines en validation: {len(val_train)}")
print(f"Forme X validation: {X_val.shape}")

Nombre de protéines en validation: 12
Forme X validation: torch.Size([2033, 13])


In [10]:
# =============================================================================
#              CHARGEMENT DES DONNÉES DE TEST
# =============================================================================

# Même processus pour les données de test
raw_data_sample = get_data('protein-secondary-structure.test')
data = parse_raw_data(raw_data_sample)
x_test, y_test = create_windows(data, window_size=13)

print(f"Nombre de protéines en test: {len(data)}")
print(f"Forme X test: {x_test.shape}")

Nombre de protéines en test: 17
Forme X test: torch.Size([3520, 13])


### 1.6 Application du One-Hot Encoding

On convertit maintenant toutes les données en format one-hot.

In [11]:
# =============================================================================
#              ENCODAGE ONE-HOT DE TOUTES LES DONNÉES
# =============================================================================
#
# On applique la fonction one_hot à chaque ensemble de données
# Cela transforme les indices [N, 13] en vecteurs [N, 273]

# Données d'entraînement
x_one_hot_train = one_hot(X_true)

# Données de validation
x_one_hot_val = one_hot(X_val)

# Données de test
x_one_hot_test = one_hot(x_test)

Shape après one-hot: torch.Size([16072, 13, 21])
Shape après flatten: torch.Size([16072, 273])
Shape après one-hot: torch.Size([2033, 13, 21])
Shape après flatten: torch.Size([2033, 273])
Shape après one-hot: torch.Size([3520, 13, 21])
Shape après flatten: torch.Size([3520, 273])


### 1.7 Transfert sur GPU (si disponible)

On vérifie si un GPU est disponible et on y copie les données pour accélérer l'entraînement.

In [12]:
# =============================================================================
#              CONFIGURATION GPU / CPU
# =============================================================================
#
# On vérifie si CUDA (GPU NVIDIA) est disponible
# Si oui, on utilise le GPU pour accélérer les calculs
# Sinon, on reste sur le CPU

# device sera soit 'cuda' soit 'cpu' selon la disponibilité
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif utilisé: {device}")

# Copie des tenseurs sur le dispositif choisi
x_one_hot_train = x_one_hot_train.to(device)
x_one_hot_val = x_one_hot_val.to(device)
x_one_hot_test = x_one_hot_test.to(device)

# On copie aussi les indices (pour le LSTM notamment)
x_test = x_test.to(device)
X_true = X_true.to(device)
X_val = X_val.to(device)

# Les labels
y_test = y_test.to(device)
y_true = y_true.to(device)
y_val = y_val.to(device)

Dispositif utilisé: cuda


---

## 3 Architecture du Modèle MLP

### Description de l'architecture

Le modèle MLP (Multi-Layer Perceptron) est un réseau de neurones feed-forward avec :

- **Entrée** : 273 dimensions (13 fenêtres × 21 one-hot)
- **Couche 1** : Linear(273 → 128) + ReLU + Dropout(0.3)
- **Couche 2** : Linear(128 → 40) + ReLU + Dropout(0.6)
- **Sortie** : Linear(40 → 3) (3 classes : Hélice, Feuillet, Coil)

**Pourquoi ces choix ?**
- Graduelle réduction de dimension (273 → 128 → 40 → 3)
- Dropout élevé (0.6) sur la couche cachée pour éviter le surapprentissage
- ReLU comme fonction d'activation (simplice et efficace)

In [13]:
# =============================================================================
#                    DÉFINITION DU MODÈLE MLP
# =============================================================================
#
# Voici notre architecture de réseau de neurones pour la classification.
# C'est un Perceptron Multicouches (MLP) classique avec :
# - Une couche d'entrée de 273 neurones (13 * 21)
# - Deux couches cachées avec dropout pour régularisation
# - Une couche de sortie de 3 neurones (pour les 3 classes)

class MLP(nn.Module):
    """
    Modèle MLP pour la prédiction de structure secondaire des protéines.
    
    Architecture:
    - Input: 273 features (13 fenêtres * 21 one-hot)
    - Hidden 1: 128 neurones + ReLU + Dropout(0.3)
    - Hidden 2: 40 neurones + ReLU + Dropout(0.6)
    - Output: 3 neurones (classification)
    """
    
    def __init__(self, window_size=13, vocab_size=21):
        """
        Initialisation du modèle.
        
        Args:
            window_size: Taille de la fenêtre glissante (défaut=13)
            vocab_size: Nombre d'acides aminés possibles (défaut=21 avec PAD)
        """
        # Appel obligatoire du constructeur parent (nn.Module)
        super().__init__()
        
        # Taille d'entrée: window_size * vocab_size = 13 * 21 = 273
        input_size = window_size * vocab_size
        
        # =================================================================
        # DÉFINITION DES COUCHES
        # =================================================================
        #
        # nn.Sequential: Permet d'enchaîner les couches automatiquement
        #
        # Couche 1: 273 → 128
        # - Linear: transformation linéaire y = Wx + b
        # - ReLU: activation max(0, x) pour la non-linéarité
        # - Dropout: désactive aléatoirement des neurones pendant l'entraînement
        #
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),    # Couche 1: entrée → 128
            nn.ReLU(),                      # Activation ReLU
            nn.Dropout(0.3),                 # Dropout 30% (régularisation)
            
            nn.Linear(128, 40),            # Couche 2: 128 → 40
            nn.ReLU(),                      # Activation ReLU
            nn.Dropout(0.6),                 # Dropout 60% (plus fort ici)
            
            nn.Linear(40, 3)               # Couche de sortie: 40 → 3
            # Pas d'activation finale car CrossEntropyLoss inclut softmax
        )

    def forward(self, x):
        """
        Passe avant (forward pass).
        
        Args:
            x: Tenseur d'entrée de forme [Batch, 273]
        
        Returns:
            Tenseur de sortie de forme [Batch, 3] (logits)
        """
        return self.model(x)

### 3.1 Initialisation aléatoire avec graine

On fixe la graine aléatoire pour reproduire les résultats.

In [14]:
# =============================================================================
#              FIXATION DE LA GRAINÉ ALÉATOIRE
# =============================================================================
#
# Cela garantit que le modèle sera initialisé de la même manière
# à chaque exécution du notebook.

seed = 3227  # Cette graine donne de bons résultats

# Fixation de la graine pour PyTorch
torch.manual_seed(seed)

# Si on utilise un GPU CUDA, on fixe aussi sa graine
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

---

## 4 Entraînement du Modèle

### Configuration de l'entraînement
- **Optimizer** : Adam avec lr=0.00028
- **Loss** : CrossEntropyLoss (classification multi-classes)
- **Epochs** : 490
- **Évaluation** : Toutes les 50 époques

In [15]:
# =============================================================================
#                    ENTRAÎNEMENT DU MODÈLE MLP
# =============================================================================
#
# Voici la boucle d'entraînement complète.
# On entraîne le modèle sur toutes les données d'un coup (full batch).
#
# Étapes par époque:
# 1. Mettre le modèle en mode entraînement (train)
# 2. Forward pass: passer les données dans le modèle
# 3. Calculer la loss (erreur)
# 4. Backward pass: calculer les gradients
# 5. Mise à jour des poids avec l'optimiseur
# 6. Évaluation sur validation toutes les 50 époques

# -----------------------------------------------------------------------------
# CRÉATION DU MODÈLE ET CONFIGURATION
# -----------------------------------------------------------------------------

# Instanciation du modèle et transfert sur le device
model = MLP().to(device)

# Optimiseur Adam (Adaptive Moment Estimation)
# - lr (learning rate): pas de mise à jour des poids
# - Adam est très populaire car il adapte automatiquement le lr
optimizer = torch.optim.Adam(model.parameters(), lr=0.00028)

# Fonction de perte pour classification multi-classes
# CrossEntropyLoss combine:
# - LogSoftmax (pour avoir des probabilités)
# - Negative Log Likelihood (pour mesurer l'erreur)
loss_fn = nn.CrossEntropyLoss()

# Nombre d'époques d'entraînement
# 490 époques suffisent pour converger avec notre lr
epochs = 490

# -----------------------------------------------------------------------------
# BOUCLE D'ENTRAÎNEMENT
# -----------------------------------------------------------------------------

for i in range(epochs):
    # === ÉTAPE 1: MODE ENTRAÎNEMENT ===
    model.train()  # Active le dropout et autres couches spécifiques entraînement
    
    # === ÉTAPE 2: FORWARD PASS ===
    # On passe toutes les données d'un coup (pas de mini-batch ici)
    # y_train contient les logits (scores non normalisés) pour chaque classe
    y_train = model(x_one_hot_train)
    
    # === ÉTAPE 3: CALCUL DE LA LOSS ===
    # On compare les prédictions aux vraies étiquettes
    loss = loss_fn(y_train, y_true)
    
    # === ÉTAPE 4 & 5: BACKWARD PASS ET OPTIMISATION ===
    optimizer.zero_grad()   # Remettre les gradients à zéro
    loss.backward()         # Calculer les gradients (backpropagation)
    optimizer.step()         # Mettre à jour les poids
    
    # === ÉTAPE 6: ÉVALUATION ===
    if i % 50 == 0:  # Toutes les 50 époques
        # Accuracy sur les données d'entraînement
        y_pred = torch.argmax(y_train, dim=1)  # Prédiction = classe avec score max
        accuracy = (y_pred == y_true).float().mean()
        print(f"Époque {i:3d} | Loss: {loss.item():.4f} | Accuracy Train: {accuracy.item()*100:.2f}%")
        
        # Accuracy sur les données de validation
        with torch.no_grad():  # Pas de calcul de gradients pour la validation
            model.eval()  # Mode évaluation (désactive dropout)
            y_val_train = model(x_one_hot_val)
            y_val_pred = torch.argmax(y_val_train, dim=1)
            accuracy_val = (y_val_pred == y_val).float().mean()
            print(f"           | Accuracy Val: {accuracy_val*100:.2f}%")
            
            # Remettre en mode entraînement pour la prochaine itération
            model.train()

Époque   0 | Loss: 1.0758 | Accuracy Train: 51.62%
           | Accuracy Val: 56.27%
Époque  50 | Loss: 0.9989 | Accuracy Train: 54.28%
           | Accuracy Val: 56.27%
Époque 100 | Loss: 0.9411 | Accuracy Train: 54.27%
           | Accuracy Val: 56.27%
Époque 150 | Loss: 0.8648 | Accuracy Train: 59.72%
           | Accuracy Val: 61.58%
Époque 200 | Loss: 0.8234 | Accuracy Train: 63.18%
           | Accuracy Val: 65.27%
Époque 250 | Loss: 0.8011 | Accuracy Train: 64.88%
           | Accuracy Val: 65.42%
Époque 300 | Loss: 0.7780 | Accuracy Train: 66.30%
           | Accuracy Val: 65.32%
Époque 350 | Loss: 0.7520 | Accuracy Train: 67.97%
           | Accuracy Val: 65.37%
Époque 400 | Loss: 0.7187 | Accuracy Train: 70.17%
           | Accuracy Val: 65.96%
Époque 450 | Loss: 0.6825 | Accuracy Train: 72.32%
           | Accuracy Val: 66.60%


---

## 5 Évaluation du Modèle

On évalue maintenant le modèle sur les données de test pour obtenir le score final.

In [16]:
# =============================================================================
#                 ÉVALUATION FINALE SUR LE TEST SET
# =============================================================================
#
# On calcule l'accuracy (pourcentage de prédictions correctes).

# Mode évaluation (désactive dropout, utilise batchnorm en mode inference)
model.eval()

# Pas de gradients nécessaires pour l'évaluation (économie de mémoire)
with torch.no_grad():
    # Forward pass sur les données de test
    y_test_train = model(x_one_hot_test)
    
    # Prédiction = argmax des logits (classe avec le score le plus élevé)
    y_test_pred = torch.argmax(y_test_train, dim=1)
    
    # Calcul de l'accuracy
    accuracy = (y_test_pred == y_test).float().mean()
    print(f"Accuracy sur le test set: {accuracy*100:.2f}%")

# Optionnel: Sauvegarder le modèle
# torch.save(model.state_dict(), 'model_v5.pth')

Accuracy sur le test set: 63.72%


---

## 6 Approche à Deux Étapes (Stage 2)

### Idée principale

Au lieu de donner les probabilitées directes au modèle 2, on utilise les **sorties du modèle 1** (après softmax) comme nouvelles caractéristiques d'entrée.

**Pourquoi ça marche ?**
- Le modèle 1 a déjà appris des motifs utiles
- Les probabilités en sortie contiennent de l'information enrichie
- Le modèle 2 peut "corriger" les erreurs du modèle 1

### Étapes:
1. **Stage 1** : Entraîner le MLP comme avant
2. **Extraire les probabilités** : Pour chaque fenêtre, obtenir P(classe)
3. **Créer de nouvelles fenêtres** : Utiliser les probs comme features (13 × 3 = 39)
4. **Stage 2** : Entraîner un nouveau MLP (FilterMLP) sur ces features

In [17]:
# =============================================================================
#          CRÉATION DES ENTRÉES POUR LE MODÈLE STAGE 2
# =============================================================================
#
# Cette fonction transforme les sorties du modèle 1 en entrées pour le modèle 2.
#
# Étapes:
# 1. Récupérer les logits du modèle 1
# 2. Convertir en probabilités (softmax)
# 3. Créer des fenêtres glissantes sur ces probabilités
#
# Résultat: Chaque résidu est décrit par 13 * 3 = 39 features
# (les probabilités des 13 positions de la fenêtre)

def create_stage2_inputs(model, x_input, window_size=13):
    """
    Crée les entrées pour le modèle Stage 2 à partir des sorties du modèle 1.
    
    Args:
        model: Le modèle MLP déjà entraîné (Stage 1)
        x_input: Les données one-hot [N, 273]
        window_size: Taille de la fenêtre (défaut=13)
    
    Returns:
        windows: Tenseur de forme [N, window_size * 3] = [N, 39]
    """
    model.eval()  # Mode évaluation
    
    with torch.no_grad():
        # A. On récupère les logits (scores bruts) du modèle 1
        logits = model(x_input)  # [N, 3]
        
        # B. On transforme en probabilités avec softmax
        probs = torch.softmax(logits, dim=1)  # [N, 3]
    
    # C. Préparation de la fenêtre glissante sur les probabilités
    num_samples = probs.shape[0]
    half_window = window_size // 2  # = 6
    
    # Padding avec des zéros (probabilité nulle) aux bords
    # Cela signifie "je ne sais pas" pour les bords
    padding = torch.zeros((half_window, 3)).to(device)
    padded_probs = torch.cat([padding, probs, padding], dim=0)  # [N + 12, 3]
    
    # =================================================================
    # CRÉATION DES FENÊTRES AVEC unfold()
# =================================================================
    # unfold permet de créer des fenêtres glissantes efficacement
    # 
    # padded_probs: [N + 12, 3]
    # unfold(0, window_size, 1):
    #   - dimension 0 (les lignes)
    #   - taille de fenêtre 13
    #   - pas de 1
    # Résultat: [N, 3, 13] -> on rearrange en [N, 39]
    
    windows = padded_probs.unfold(0, window_size, 1)  # [N, 3, 13]
    
    # Remettre dans l'ordre: [N, 13, 3] puis aplatir en [N, 39]
    windows = windows.permute(0, 2, 1).reshape(num_samples, -1)
    
    return windows

print("Génération des entrées pour le Modèle Stage 2...")

# Application aux trois ensembles de données
# Note: y_true, y_val, y_test RESTENT LES MÊMES
X2_train = create_stage2_inputs(model, x_one_hot_train)
X2_val   = create_stage2_inputs(model, x_one_hot_val)
X2_test  = create_stage2_inputs(model, x_one_hot_test)

print(f"Forme des données Stage 2 (Train): {X2_train.shape}")  # Devrait être [N, 39]

Génération des entrées pour le Modèle Stage 2...
Forme des données Stage 2 (Train): torch.Size([16072, 39])


### 6.1 Architecture du FilterMLP (Stage 2)

Un MLP plus léger qui prend les probabilités du modèle 1 en entrée.

In [18]:
# =============================================================================
#               DÉFINITION DU MODÈLE FILTER MLP (STAGE 2)
# =============================================================================
#
# Ce modèle est plus léger car il reçoit des features déjà "traitées"
# par le modèle 1 (les probabilités au lieu des one-hot bruts).
#
# Architecture:
# - Entrée: 39 features (13 positions * 3 classes de probabilité)
# - Hidden: 64 → 32 neurones
# - Sortie: 3 classes

class FilterMLP(nn.Module):
    """
    Modèle Stage 2: Filtre qui affine les prédictions du modèle 1.
    
    Il prend les probabilités du modèle 1 comme entrée et cherche
    à corriger les erreurs en regardant le contexte local.
    """
    
    def __init__(self, window_size=13):
        super().__init__()
        
        # Dimension d'entrée: window_size * 3 = 13 * 3 = 39
        input_dim = window_size * 3
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),   # 39 → 64
            nn.ReLU(),
            nn.Dropout(0.5),             # 50% dropout
            
            nn.Linear(64, 32),           # 64 → 32
            nn.ReLU(),
            nn.Dropout(0.3),             # 30% dropout
            
            nn.Linear(32, 3)             # 32 → 3 (sortie)
        )
        
    def forward(self, x):
        return self.model(x)

In [19]:
# =============================================================================
#           INITIALISATION DU MODÈLE STAGE 2
# =============================================================================

# Nouvelle graine pour l'initialisation aléatoire
seed = 8601
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Création du modèle Stage 2
model2 = FilterMLP().to(device)

# Optimiseur avec un lr différent (plus élevé car plus petit modèle)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.00193)

# Fonction de perte
loss_fn = nn.CrossEntropyLoss()

# Nombre d'époques (moins que le modèle 1 car on part d'une bonne init)
epochs_stage2 = 40

print("Démarrage de l'entraînement du Correcteur (Stage 2)...")

Démarrage de l'entraînement du Correcteur (Stage 2)...


### 6.2 Entraînement du Stage 2

In [20]:
# =============================================================================
#              ENTRAÎNEMENT DU MODÈLE STAGE 2
# =============================================================================

# Liste pour suivre l'évolution des accuracies
train_acc_history = []
val_acc_history = []

for i in range(epochs_stage2):
    model2.train()
    
    # === FORWARD ===
    y2_pred_logits = model2(X2_train)
    loss = loss_fn(y2_pred_logits, y_true)
    
    # === BACKWARD ===
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()
    
    # === ÉVALUATION (toutes les 5 époques) ===
    if i % 5 == 0:
        # Accuracy sur train
        pred_train = torch.argmax(y2_pred_logits, dim=1)
        acc_train = (pred_train == y_true).float().mean()
        train_acc_history.append(acc_train.item())
        
        # Accuracy sur validation
        model2.eval()
        with torch.no_grad():
            y2_val_logits = model2(X2_val)
            pred_val = torch.argmax(y2_val_logits, dim=1)
            acc_val = (pred_val == y_val).float().mean()
            val_acc_history.append(acc_val.item())
        
        print(f"Époque {i:3d} | Loss: {loss.item():.4f} | Acc Train: {acc_train*100:.2f}% | Acc Val: {acc_val*100:.2f}%")
        
        model2.train()  # Retour en mode train

Époque   0 | Loss: 1.0872 | Acc Train: 37.11% | Acc Val: 55.04%
Époque   5 | Loss: 1.0193 | Acc Train: 54.21% | Acc Val: 56.27%
Époque  10 | Loss: 0.9637 | Acc Train: 54.28% | Acc Val: 56.27%
Époque  15 | Loss: 0.9224 | Acc Train: 54.32% | Acc Val: 56.27%
Époque  20 | Loss: 0.8640 | Acc Train: 55.69% | Acc Val: 56.27%
Époque  25 | Loss: 0.7980 | Acc Train: 64.38% | Acc Val: 62.81%
Époque  30 | Loss: 0.7362 | Acc Train: 68.84% | Acc Val: 64.54%
Époque  35 | Loss: 0.6882 | Acc Train: 71.32% | Acc Val: 67.29%


### 6.3 Évaluation Finale du Model

In [21]:
# =============================================================================
#           ÉVALUATION FINALE DU SYSTÈME COMPLET (STAGE 2)
# =============================================================================

print("\n" + "="*50)

model2.eval()
with torch.no_grad():
    # Prédiction sur le test set
    y2_test_logits = model2(X2_test)
    y2_test_pred = torch.argmax(y2_test_logits, dim=1)
    
    # Accuracy finale
    final_acc = (y2_test_pred == y_test).float().mean()

print(f"SCORE FINAL (Stage 2): {final_acc*100:.2f}%")
print("="*50)


SCORE FINAL (Stage 2): 65.20%


### 6.4 Analyse détaillée avec Matrice de Confusion

In [22]:
# =============================================================================
#              ANALYSE DÉTAILLÉE: MATRICE DE CONFUSION
# =============================================================================
#
# La matrice de confusion montre où le modèle se trompe.
# - Lignes: vraies classes
# - Colonnes: classes prédites
# - Diagonale: bonnes prédictions

from sklearn.metrics import classification_report

# Affichage du rapport de classification
print(classification_report(y_test.cpu(), y2_test_pred.cpu(), 
                             target_names=['Helix', 'Sheet', 'Coil']))

# Optionnel: Sauvegarder le modèle Stage 2
# torch.save(model2.state_dict(), 'model_v5.pth')

              precision    recall  f1-score   support

       Helix       0.55      0.54      0.54       849
       Sheet       0.58      0.28      0.38       748
        Coil       0.70      0.85      0.77      1923

    accuracy                           0.65      3520
   macro avg       0.61      0.56      0.56      3520
weighted avg       0.64      0.65      0.63      3520



---

### Fonction de recherche des hyperparamètres

In [23]:
"""
import torch
import torch.nn as nn
import numpy as np
import random
import copy
import time

# --- CONFIGURATION DE LA BATAILLE ---
# Nombre total d'essais (Tu peux mettre 100 si tu as le temps)
NUM_TRIALS = 50 

# Patience pour tuer les modèles nuls rapidement
PATIENCE = 15 
MAX_EPOCHS = 100 # Pas besoin de 900, l'early stopping coupera avant

# Variables pour stocker le GRAND VAINQUEUR
global_best_acc = 0.0
global_best_model_state = None
global_best_params = {}

print(f" DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    
    # 1. TIRAGE AU SORT DES ARMES (Hyperparamètres)
    # Graine aléatoire entre 0 et 10000
    seed = random.randint(0, 10000)
    
    # Learning Rate aléatoire sur échelle Logarithmique (entre 0.0001 et 0.01)
    # C'est mieux que l'aléatoire normal car ça explore les petits et les gros ordres de grandeur
    lr_log = np.random.uniform(-4, -2) # -4 = 10^-4, -2 = 10^-2
    lr = 10 ** lr_log
    
    # 2. APPLICATION DE LA GRAINE (Reproductibilité pour ce trial)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU COMBATTANT
    # Important : On recrée le modèle de zéro à chaque fois !
    model = MLP().to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    # Variables locales pour ce round
    best_val_acc_round = 0.0
    patience_counter = 0
    epoch_stop = 0
    
    # --- COMBAT (Entraînement) ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        
        # Train Loop
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()
            
        # --- VALIDATION (Juge de paix) ---
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)
                logits_val = model(x_val)
                preds = torch.argmax(logits_val, dim=1)
                correct += (preds == y_val).sum().item()
                total += y_val.size(0)
        
        val_acc = correct / total
        
        # --- EARLY STOPPING LOGIC ---
        if val_acc > best_val_acc_round:
            best_val_acc_round = val_acc
            patience_counter = 0
            # Si c'est un potentiel champion global, on sauvegarde ses poids en RAM temporairement
            # (Pour éviter de copier des poids inutiles, on ne copie que si c'est très bon)
            if val_acc > global_best_acc:
                temp_best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            epoch_stop = epoch
            break # K.O. Technique
            
    # --- FIN DU ROUND ---
    # Est-ce que ce combattant est le nouveau champion du monde ?
    if best_val_acc_round > global_best_acc:
        print(f"[Trial {trial}/{NUM_TRIALS}] NOUVEAU RECORD ! Acc: {best_val_acc_round*100:.2f}% | Seed: {seed} | LR: {lr:.5f}")
        global_best_acc = best_val_acc_round
        global_best_model_state = temp_best_state
        global_best_params = {'seed': seed, 'lr': lr, 'epoch': epoch_stop}
    else:
        # On affiche juste un petit point ou log pour dire qu'on avance
        # print(f"   [Trial {trial}] Échoué (Best: {best_val_acc_round*100:.2f}%)")
        pass

# --- PODIUM FINAL ---
total_time = time.time() - start_time
print("\n" + "="*60)
print(f"BATTLE TERMINÉE en {total_time/60:.1f} minutes.")
print(f"LE GRAND VAINQUEUR EST :")
print(f"   - Accuracy Validation : {global_best_acc*100:.2f}%")
print(f"   - Seed Magique        : {global_best_params['seed']}")
print(f"   - Learning Rate       : {global_best_params['lr']:.6f}")
print("="*60)
"""

'\nimport torch\nimport torch.nn as nn\nimport numpy as np\nimport random\nimport copy\nimport time\n\n# --- CONFIGURATION DE LA BATAILLE ---\n# Nombre total d\'essais (Tu peux mettre 100 si tu as le temps)\nNUM_TRIALS = 50 \n\n# Patience pour tuer les modèles nuls rapidement\nPATIENCE = 15 \nMAX_EPOCHS = 100 # Pas besoin de 900, l\'early stopping coupera avant\n\n# Variables pour stocker le GRAND VAINQUEUR\nglobal_best_acc = 0.0\nglobal_best_model_state = None\nglobal_best_params = {}\n\nprint(f" DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS")\nprint("="*60)\n\nstart_time = time.time()\n\nfor trial in range(1, NUM_TRIALS + 1):\n    \n    # 1. TIRAGE AU SORT DES ARMES (Hyperparamètres)\n    # Graine aléatoire entre 0 et 10000\n    seed = random.randint(0, 10000)\n    \n    # Learning Rate aléatoire sur échelle Logarithmique (entre 0.0001 et 0.01)\n    # C\'est mieux que l\'aléatoire normal car ça explore les petits et les gros ordres de grandeur\n    lr_log = np.random.uniform(-

## Résumé des Résultats

| Modèle | Accuracy Test |
|--------|----------------|
| MLP (Stage 1) | 63,72% |
| MLP + FilterMLP (Stage 2) | **65.20%** |
